
# Introduction to Metabolomics

Metabolomics is the comprehensive study of small molecules (metabolites) in biological systems. It provides insights into biochemical activities and metabolic pathways.

### Types of Metabolomics Data
- **Targeted**: Focuses on quantifying a predefined set of metabolites.
- **Untargeted**: Aims to detect and quantify as many metabolites as possible without bias.

### Data Formats
- Common formats include `.csv`, `.tsv`, `.xlsx`, and vendor-specific formats.
- Data typically includes metabolite intensities across samples.
- Spectral data may be in formats like `.mzML`, `.mzXML`, `.mgf` or `.CDF`.

### Measurement Considerations
- Sample preparation and instrument calibration are crucial.
- Batch effects and missing values can impact analysis.

---



# Hands-On: Correlation Analysis Between Metabolites and Microbes

In this exercise, we will use a small example dataset from the HMP2 cohort to:
1. Load metabolite and microbial abundance data
2. Preprocess the data (e.g., normalization, filtering, transformation)
3. Calculate Spearman correlations
4. Visualize significant correlations in a heatmap

This analysis will be performed using R.


In [ ]:

%%R
# Load required libraries
library(tidyverse)

library(reshape2)
library(ggplot2)


ERROR: Error in library(tidyverse): there is no package called ‘tidyverse’


In [ ]:

%%R
# Simulate small example datasets
set.seed(123)
metabolites <- data.frame(
  sample_id = paste0("S", 1:10),
  Met1 = rnorm(10),
  Met2 = rnorm(10),
  Met3 = rnorm(10)
)

microbes <- data.frame(
  sample_id = paste0("S", 1:10),
  MicrobeA = rnorm(10),
  MicrobeB = rnorm(10),
  MicrobeC = rnorm(10)
)

# Merge datasets
merged <- inner_join(metabolites, microbes, by = "sample_id")


In [ ]:
%% R 

# DATA FILTERING AND TRANSFORMATION

# Delete columns that have <10% of non-zero values
microbiome <- microbiome[, colSums(microbiome != 0) >= 0.1 * nrow(microbiome)]
# Apply CLR transformation (row-wise)
microbiome_clr <- as.data.frame(t(apply(microbiome, 1, function(x) clr(x))))

# Apply LOG2 transformation (row-wise)
metabolites_log <- as.data.frame(t(apply(metabolites, 1, function(x) log2(x))))


In [ ]:
%%R

# Combine the transformed data
combined_data <- cbind(metabolites_log, microbiome_clr)

# Compute Spearman correlation and adjust p-values using Benjamini-Hochberg
cor_results <- corr.test(combined_data, method = "spearman", adjust = "BH")

# Correlation coefficients
cor_matrix <- cor_results$r

# Adjusted p-values
p_matrix <- cor_results$p


# Get column names
metabolite_names <- colnames(metabolites_log)
microbe_names <- colnames(microbiome_clr)

# Subset correlation and p-value matrices
cor_subset <- cor_matrix[metabolite_names, microbe_names]
p_subset <- p_matrix[metabolite_names, microbe_names]


ERROR: Error in parse(text = x, srcfile = src): <text>:2:1: unexpected SPECIAL
1: 
2: %%
   ^


In [ ]:
%%R

# Define significance threshold
sig_threshold <- 0.05

# Create mask for significant correlations
sig_mask <- p_subset < sig_threshold

# Extract significant correlations
sig_correlations <- cor_subset[sig_mask]


In [ ]:
%%R 

# Mask insignificant correlations
cor_filtered <- cor_subset
cor_filtered[p_subset >= 0.05] <- NA  # Set non-significant to NA

# Melt and plot
cor_melted <- melt(cor_filtered, na.rm = TRUE)

#adjust figsize
options(repr.plot.width = 20, repr.plot.height = 8)

ggplot(cor_melted, aes(x = Var2, y = Var1, fill = value)) +
  geom_tile() +
  scale_fill_gradient2(low = "blue", high = "red", mid = "white",
                       midpoint = 0, limit = c(-1,1), space = "Lab",
                       name="Spearman\nCorrelation") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1, size = 12),
        axis.text.y = element_text(size = 12),
        axis.title.x = element_text(size = 16, face = "bold"),
        axis.title.y = element_text(size = 16, face = "bold")
        ) +
  labs(x = "Microbial species", y = "Metabolites", fontsize = 16)

  #print to file 
ggsave("spearman_correlation_heatmap.png", width = 20, height = 8, dpi = 300)

### Question
- What are the main challenges in untargeted metabolomics compared to targeted approaches?
- Why is normalization and transformation important in metabolomics data analysis? What methods can be used?
- Which metabolite shows the strongest overall correlation with microbes?
- How would you interpret a correlation coefficient of 0.85 between a metabolite and a microbe?
- What additional metadata (e.g., diet, disease status, age, BMI, antibiotic usage) would you want to include in your analysis?
